In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

gfa_renames = {
    "largest_property_use_type_1": "largest_use_gfa_sqft",
    "_2nd_largest_property_use_1": "second_use_gfa_sqft",
    "_3rd_largest_property_use_1": "third_use_gfa_sqft",
}

df = pd.read_csv(
    "data/nyc_2024_data.csv.gz",
    dtype={"postal_code": str, **{c: str for c in gfa_renames}},
    na_values=["Not Available"],
    low_memory=False,
)

for col in gfa_renames:
    df[col] = pd.to_numeric(df[col].str.replace(",", ""), errors="coerce")
df = df.rename(columns=gfa_renames)

df.shape

(39090, 256)

In [2]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

summary = pd.DataFrame({
    "dtype": df.dtypes,
    "n_unique": df.nunique(),
    "n_missing": df.isna().sum(),
    "examples": [df[c].dropna().unique()[:5].tolist() for c in df.columns],
})
summary.sort_values("n_unique")

,dtype,n_unique,n_missing,examples
barracks_gross_floor_area,float64,0,39090,[]
kerosene_use_kbtu,float64,0,39090,[]
data_center_pue,float64,0,39090,[]
report_year,int64,1,0,[2024]
propane_use_kbtu,float64,1,39089,[92261.0]
data_center_pdu_output_meter,float64,1,38813,[0.0]
estimated_data_flag_fuel_3,str,1,38160,[No]
eligible_for_certification_for_report_ped_y_n_,str,1,0,[No]
national_median_energy_star,int64,1,0,[50]
institutional_property_y_n_,str,1,39089,[No]


In [3]:
#Dropping columns that have the same value in every row or no values at all
constant_cols = df.columns[df.nunique(dropna=False) <= 1]
df = df.drop(columns=constant_cols)
print(f"Dropped {len(constant_cols)} columns: {list(constant_cols)}")

Dropped 10 columns: ['report_year', 'year_ending', 'national_median_energy_star', 'eligible_for_certification_for_report_ped_y_n_', 'kerosene_use_kbtu', 'data_center_pue', 'barracks_gross_floor_area', 'aggregate_meter_s_natural_gas_number_of_individual_meters_included', 'aggregate_meter_s_district_steam_number_of_individual_meters_included', 'report_generation_date']


In [4]:
#Dropping columns where at least 98% of the rows are empty
missing_share = df.isna().mean()
mostly_empty = missing_share[missing_share > 0.98].index
print(f"Dropping {len(mostly_empty)} columns: {list(mostly_empty)}")
df = df.drop(columns=mostly_empty)

Dropping 96 columns: ['energy_star_certification', 'energy_star_certification_1', 'energy_star_certification_application_status', 'fuel_oil_1_use_kbtu', 'fuel_oil_5_6_use_kbtu', 'diesel_2_use_kbtu', 'propane_use_kbtu', 'district_hot_water_use_kbtu', 'district_chilled_water_use', 'electricity_use_generated', 'electricity_use_generated_1', 'electricity_use_generated_2', 'electricity_use_generated_3', 'green_power_onsite_kwh', 'avoided_emissions_onsite', 'percent_of_recs_retained', 'percent_of_total_electricity', 'leed_us_project_id', 'estimated_data_flag_1', 'estimated_data_flag_fuel', 'estimated_data_flag_diesel', 'estimated_data_flag_2', 'data_center_ups_output_meter', 'data_center_pdu_input_meter', 'data_center_pdu_output_meter', 'data_center_it_equipment', 'data_center_it_site_energy', 'data_center_it_source_energy', 'data_center_national_median', 'data_center_gross_floor_area', 'data_center_ups_system', 'data_center_it_energy', 'data_center_energy_estimates', 'data_center_cooling_eq

In [5]:
#Dropping most of the identifier columns, keeping one as index
df = df.set_index("property_id")
df = df.drop(columns=["property_name", "nyc_building_identification", "address_1"])

In [6]:
#Checking for duplicate columns
from itertools import combinations

def find_duplicate_pairs(df, threshold=0.99, min_overlap=100):
    results = []
    for a, b in combinations(df.columns, 2):
        both = df[a].notna() & df[b].notna()
        n = both.sum()
        if n < min_overlap:
            continue
        x, y = df.loc[both, a], df.loc[both, b]
        numeric = (pd.api.types.is_numeric_dtype(x) and pd.api.types.is_numeric_dtype(y)
                   and not pd.api.types.is_bool_dtype(x) and not pd.api.types.is_bool_dtype(y))
        if numeric:
            match = np.isclose(x, y, rtol=0.001).mean()
            corr = x.corr(y, method="spearman") if x.std() > 0 and y.std() > 0 else np.nan
        else:
            match = (x.astype(str) == y.astype(str)).mean()
            corr = np.nan
        if match >= threshold or (not np.isnan(corr) and abs(corr) >= threshold):
            results.append({
                "col_a": a, "col_b": b, "overlap": n,
                "match": round(match, 4),
                "corr": round(corr, 4) if not np.isnan(corr) else None,
                "missing_a": df[a].isna().sum(), "missing_b": df[b].isna().sum(),
            })
    return pd.DataFrame(results).sort_values(["match", "corr"], ascending=False)

dupes = find_duplicate_pairs(df)
dupes

,col_a,col_b,overlap,match,corr,missing_a,missing_b
58,estimated_data_flag_fuel_1,estimated_data_flag_fuel_3,643,1.0000,NaN,32593,38160
59,estimated_data_flag_fuel_2,estimated_data_flag_fuel_3,325,1.0000,NaN,37116,38160
72,indoor_water_use_all_water,municipally_supplied_potable_2,18382,0.9998,1.0000,20708,20707
69,water_use_all_water_sources,municipally_supplied_potable_1,18616,0.9996,1.0000,20470,20472
73,municipally_supplied_potable_1,municipally_supplied_potable_2,18375,0.9991,0.9999,20472,20707
68,water_use_all_water_sources,indoor_water_use_all_water,18373,0.9989,0.9999,20470,20708
71,indoor_water_use_all_water,municipally_supplied_potable_1,18374,0.9989,0.9999,20708,20472
56,alert_property_has_no_uses,alert_gross_floor_area_is,39090,0.9988,NaN,0,0
70,water_use_all_water_sources,municipally_supplied_potable_2,18373,0.9987,0.9999,20470,20707
6,metered_areas_energy,metered_areas_water,22961,0.9985,NaN,8027,15980


In [7]:
#Dropping duplicates
df = df.drop(columns=[
    "indoor_water_use_all_water", "municipally_supplied_potable_1", "municipally_supplied_potable_2",
    "electricity_use_grid_purchase_1", "electricity_use_grid_purchase_2", "electricity_use_grid_purchase_3",
    "natural_gas_use_therms_",
    "primary_property_type", "largest_property_use_type",
    "property_gfa_calculated", "property_gfa_calculated_1",
    "estimated_data_flag_fuel_3",
    "weather_normalized_site_eui", "weather_normalized_site_energy",
    "weather_normalized_source", "weather_normalized_source_1",
    "natural_gas_weather_normalized", "electricity_weather_normalized",
    "number_of_active_energy_meters_1",
])

In [9]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

summary = pd.DataFrame({
    "dtype": df.dtypes,
    "n_unique": df.nunique(),
    "n_missing": df.isna().sum(),
    "examples": [df[c].dropna().unique()[:5].tolist() for c in df.columns],
})
summary.sort_values("n_unique")

,dtype,n_unique,n_missing,examples
alert_data_center_issue_with,str,2,0,"[Ok, Possible Issue]"
multifamily_housing_government,str,2,25385,"[No, 100% Yes]"
estimated_data_flag_fuel_2,str,2,37116,"[No, Yes]"
estimated_data_flag_fuel_1,str,2,32593,"[No, Yes]"
estimated_data_flag_district,str,2,37513,"[No, Yes]"
estimated_data_flag_natural,str,2,9533,"[No, Yes]"
estimated_data_flag,str,2,8361,"[Yes, No]"
temporary_values,bool,2,0,"[False, True]"
alert_energy_no_meters,str,2,0,"[Ok, Possible Issue]"
alert_property_has_no_uses,str,2,0,"[Ok, Possible Issue]"
